# Работаем с LLM с помощью библиотеки openai

В этом тюториале мы научимся общаться с LLM через т.н. OpenAI API.

Работа с OpenAI API уже реализована в библиотеке _openai_: https://github.com/openai/openai-python

Эту библиотеку можно поставить в виртуальное окружение командой `pip install openai`

Импортируем модули которые нам понадобятся впоследствии:

In [ ]:
import json
import logging
from timeit import default_timer as timer

from openai import OpenAI

Подготовим логгер:

In [ ]:
logging.basicConfig(level=logging.INFO)

## Делаем запросы к LLM

Загрузим параметры соединения из файла-конфига:

In [ ]:
# Default config allows to connect to the local LLM launched with vllm.sh script
#config_file = "openai-config.default.json"
config_file = "openai-config.custom.json"

# Load
with open(config_file, "rt") as f:
    openai_config = json.load(f)
    print(openai_config)

Создадим клиента:

In [ ]:
client = OpenAI(api_key=openai_config["api_key"], base_url=openai_config["api_base_url"], max_retries=0)
logging.info(f"created OpenAI client")

Подготовим системный и пользовательский промпты:

In [ ]:
system_prompt = "You are a helpful assistant"
user_prompt = "What are some of the places to visit around Barcelona?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

Зададим дополнительные параметры. В частности, явно скажем модели что хотим использовать reasoning:

In [ ]:
extra_body = {}
extra_body["chat_template_kwargs"] = { "enable_thinking": True }

Подадим запрос к модели, используя _completions API_:

In [ ]:
start = timer()
response = client.chat.completions.create(
    model=openai_config["model"],
    messages=messages,
    extra_body=extra_body,
    seed=22,
    #max_tokens=1000,
)
elapsed = timer() - start
print(response)

Распакуем ответ:

In [ ]:
# Extract reasoning and content
completion_message = response.choices[0].message
reasoning_content = completion_message.reasoning
content = completion_message.content

# Token usage
num_input_tokens = response.usage.prompt_tokens
num_output_tokens = response.usage.completion_tokens
tps = 1. / (elapsed / num_output_tokens)

# Show results
print("PROMPT:\n")
print(user_prompt)
print("REASONING:\n") 
print(reasoning_content)
print("CONTENT:\n")
print(content)
logging.info(f"got response: num_input_tokens = {num_input_tokens} num_output_tokens = {num_output_tokens}"
             f" tps = {tps:.2f} elapsed = {elapsed:.3f}")

Таким образом, мы научились подавать запросы к модели используя библиотеку _openai_.